# Extracting Structure from Chaos: Insurance Claims with LangGraph + Snowflake Cortex

In this hands-on lab, you'll build a production-grade pipeline to extract structured data from unstructured insurance adjuster notes using **LangGraph** for orchestration and **Snowflake Cortex** for LLM inference.

## Learning Objectives

1. Load and explore the FEMA NFIP flood claims dataset in Snowflake
2. Design extraction schemas with Pydantic -- including fields that may not exist in the source text
3. Organize fields into logical groups for focused, parallel extraction
4. Build a LangGraph pipeline using the **Send API** for fan-out extraction with validate → fix → finalize
5. Run batch extraction with checkpointing and restartability
6. Measure extraction accuracy against ground truth

---
# Section 1: Exploring the FEMA NFIP Data

The [FEMA National Flood Insurance Program (NFIP)](https://www.fema.gov/about/openfema/data-sets#nfip) dataset contains over **2.7 million flood insurance claims** across the United States. Each row includes structured fields like damage amounts, property details, geographic info, and flood characteristics.

Our plan:
1. Explore the data to understand what we're working with
2. Create a curated sample for the lab exercises


### EXERCISE: Explore the Data

Before we start generating text and extracting data, you need to understand what you're working with. Write SQL queries to answer these questions:

1. What are the **top 10 states** by claim count?
2. What is the **average building damage amount** grouped by flood event?
3. What are the **distinct flood zone** values and how many claims are in each?
4. What is the **range of water depth** values (min, max, avg)?

Understanding these distributions will help you evaluate your extraction pipeline's accuracy later.

In [ ]:
USE DATABASE CLAIMS_EXTRACTION_LAB;
USE SCHEMA HOL;

In [ ]:
-- EXERCISE: Write your exploration queries here.
-- Hint: Use GROUP BY, COUNT(*), AVG(), MIN(), MAX()

-- 1. Top 10 states by claim count
-- SELECT state, COUNT(*) AS claim_count
-- FROM CLAIMS_DATA
-- GROUP BY state
-- ORDER BY claim_count DESC
-- LIMIT 10;


-- 2. Average building damage by flood event (top 10 events)
-- SELECT flood_event, AVG(building_damage_amount) AS avg_building_damage
-- FROM CLAIMS_DATA
-- WHERE flood_event IS NOT NULL
-- GROUP BY flood_event
-- ORDER BY avg_building_damage DESC
-- LIMIT 10;


-- 3. Flood zone distribution
-- SELECT flood_zone, COUNT(*) AS count
-- FROM CLAIMS_DATA
-- GROUP BY flood_zone
-- ORDER BY count DESC;


-- 4. Water depth statistics
-- SELECT MIN(water_depth_inches), MAX(water_depth_inches), AVG(water_depth_inches)
-- FROM CLAIMS_DATA
-- WHERE water_depth_inches IS NOT NULL;


### Setup

Load the Python libraries we'll use throughout the lab and connect to your Snowflake session.

In [ ]:
import json
import time
import operator
from typing import TypedDict, Literal, Optional, Annotated
from datetime import datetime

from pydantic import BaseModel, Field, field_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print(f"Connected to Snowflake: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

### Create a Lab Sample

We'll work with a small sample of claims that have non-null values for key fields. This keeps iteration fast while ensuring we have meaningful data to extract.

In [ ]:
-- Create a small sample for iterative testing during the lab

CREATE OR REPLACE TABLE CLAIMS_SAMPLE AS
SELECT
    *
FROM (
    SELECT *
    FROM FIMA_CLAIMS
    WHERE building_damage_amount > 0
      AND contents_damage_amount > 0
      AND flood_event IS NOT NULL
      AND water_depth IS NOT NULL
      AND state IS NOT NULL
      AND flood_zone IS NOT NULL
    LIMIT 5
);

SELECT * FROM CLAIMS_SAMPLE;

---
# Section 2: Designing Extraction Schemas with Pydantic

In real-world extraction, you rarely have clean 1:1 alignment between your schema and the source text. Some fields will be explicitly stated, others require interpretation, and some might not be present at all. A good schema handles all three cases.

**Where you are in the overall pipeline:**

```
Adjuster Notes → [ Pydantic Schema ] → [ LangGraph Pipeline ] → [ Batch Extraction ] → Results
                       ↑ YOU ARE HERE
```

We'll use **Pydantic models** to:

1. Define target schemas with types, descriptions, and **field groups**
2. Generate JSON schemas for Cortex **structured output**
3. Validate extracted data and catch errors automatically

We're expanding beyond the obvious fields to ~21 total, organized into three groups:
- **Claim & Location**: identifiers and geographic info
- **Flood Event & Damage**: what happened and what it cost
- **Property Details**: building characteristics and conditions

Some fields (like `mold_present` or `structural_damage_severity`) may be inferable from narrative observations. Others (like `elevation_certificate` or `building_description_code`) exist in the underlying dataset but were never mentioned in the adjuster notes -- the LLM should return null for these. This is realistic: you always define your ideal schema, then see what the source text can actually provide.

In [ ]:
# --- PROVIDED: These 21 fields are your extraction target ---
# Each field has a type and description -- the descriptions help the LLM
# understand what to look for in the unstructured text.

class ClaimExtraction(BaseModel):
    """Structured data extracted from insurance adjuster field notes."""
    
    # --- Group 1: Claim & Location ---
    claim_id: Optional[str] = Field(None, description="The claim identifier or number")
    date_of_loss: Optional[str] = Field(None, description="Date when the flood damage occurred (YYYY-MM-DD format)")
    state: Optional[str] = Field(None, description="Two-letter US state code (e.g., TX, FL, LA)")
    county_code: Optional[str] = Field(None, description="County FIPS code")
    flood_zone: Optional[str] = Field(None, description="FEMA flood zone designation (e.g., AE, V, X, A)")
    
    # --- Group 2: Flood Event & Damage ---
    flood_event: Optional[str] = Field(None, description="Name of the flood event or disaster")
    water_depth_inches: Optional[int] = Field(None, description="Depth of flood water in inches")
    flood_duration_hours: Optional[int] = Field(None, description="How long flood water remained, in hours")
    building_damage_amount: Optional[float] = Field(None, description="Dollar amount of building/structural damage")
    contents_damage_amount: Optional[float] = Field(None, description="Dollar amount of contents/personal property damage")
    building_property_value: Optional[float] = Field(None, description="Assessed or estimated value of the building")
    contents_property_value: Optional[float] = Field(None, description="Assessed or estimated value of contents")
    structural_damage_severity: Optional[str] = Field(None, description="Overall severity of structural damage: minor, moderate, severe, or catastrophic")
    amount_paid_building: Optional[float] = Field(None, description="Insurance payout amount for building damage")
    amount_paid_contents: Optional[float] = Field(None, description="Insurance payout amount for contents damage")
    
    # --- Group 3: Property Details ---
    basement_type: Optional[int] = Field(None, description="Basement/enclosure/crawlspace type code (0-4)")
    floodproofed: Optional[bool] = Field(None, description="Whether the building has floodproofing measures")
    mold_present: Optional[bool] = Field(None, description="Whether mold was observed during inspection")
    number_of_stories: Optional[int] = Field(None, description="Number of stories in the building")
    elevation_certificate: Optional[bool] = Field(None, description="Whether an elevation certificate exists for the property")
    building_description_code: Optional[str] = Field(None, description="Building description type code (e.g., 1=single family, 2=2-4 family)")

print(f"ClaimExtraction schema: {len(ClaimExtraction.model_fields)} fields defined")

In [ ]:
# --- PROVIDED: Schema utilities (you will call schema_for_group() later) ---
# Organize fields into groups for focused extraction prompts.
# Each group gets its own Cortex call with a tailored prompt.
FIELD_GROUPS = {
    "claim_location": {
        "label": "Claim & Location",
        "fields": ["claim_id", "date_of_loss", "state", "county_code", "flood_zone"],
    },
    "flood_damage": {
        "label": "Flood Event & Damage",
        "fields": [
            "flood_event", "water_depth_inches", "flood_duration_hours",
            "building_damage_amount", "contents_damage_amount",
            "building_property_value", "contents_property_value",
            "structural_damage_severity", "amount_paid_building", "amount_paid_contents",
        ],
    },
    "property_details": {
        "label": "Property Details",
        "fields": [
            "basement_type", "floodproofed", "mold_present",
            "number_of_stories", "elevation_certificate", "building_description_code",
        ],
    },
}

ALL_FIELDS = [f for g in FIELD_GROUPS.values() for f in g["fields"]]
print(f"Extraction schema: {len(ALL_FIELDS)} fields in {len(FIELD_GROUPS)} groups")
for name, group in FIELD_GROUPS.items():
    print(f"  {group['label']}: {len(group['fields'])} fields")


# Generate the JSON schema and simplify for Cortex compatibility.
# Pydantic v2 emits anyOf for Optional fields, but Cortex needs a type array like ["string", "null"].
def simplify_schema_for_cortex(schema: dict) -> dict:
    """Convert Pydantic JSON schema to Cortex-compatible format with nullable types."""
    simplified = {"type": "object", "properties": {}}
    for field_name, field_def in schema.get("properties", {}).items():
        prop = {}
        if "anyOf" in field_def:
            types = [opt["type"] for opt in field_def["anyOf"] if "type" in opt]
            prop["type"] = types if len(types) > 1 else types[0]
        elif "type" in field_def:
            prop["type"] = field_def["type"]
        if "description" in field_def:
            prop["description"] = field_def["description"]
        simplified["properties"][field_name] = prop
    return simplified


def schema_for_group(group_key: str) -> dict:
    """Build a Cortex-compatible JSON schema for a single field group."""
    full_schema = ClaimExtraction.model_json_schema()
    group_fields = FIELD_GROUPS[group_key]["fields"]
    full_simplified = simplify_schema_for_cortex(full_schema)
    return {
        "type": "object",
        "properties": {
            k: v for k, v in full_simplified["properties"].items()
            if k in group_fields
        }
    }


extraction_schema = simplify_schema_for_cortex(ClaimExtraction.model_json_schema())
print(f"\nFull schema has {len(extraction_schema['properties'])} properties")
print("Group schemas:")
for gk in FIELD_GROUPS:
    gs = schema_for_group(gk)
    print(f"  {gk}: {len(gs['properties'])} properties")

### Test: Single Extraction with Cortex

Before building the full pipeline, let's verify that Cortex can extract data from a single note using our schema. The `response_format` parameter constrains the LLM to return valid JSON matching our schema definition.

In [ ]:
# Test the schema with a single extraction call using Cortex structured output.
# The response_format parameter forces the LLM to return valid JSON matching our schema.

sample_note = session.sql("SELECT adjuster_notes FROM CLAIMS_SAMPLE LIMIT 1").collect()[0][0]
print("Input note (first 300 chars):")
print(sample_note[:300] + "...\n")

extraction_prompt = json.dumps([
    {
        "role": "system", 
        "content": "Extract structured insurance claim data from the adjuster notes. \
                    Return all fields you can find. Use null for fields not mentioned in the notes."
    },
    {"role": "user", "content": sample_note}
])

schema_json = json.dumps(extraction_schema)

# Inspect the raw Cortex response structure
raw_result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        PARSE_JSON($${extraction_prompt}$$),
        {{
            'temperature': 0,
            'response_format': {{
                'type': 'json',
                'schema': PARSE_JSON($${schema_json}$$)
            }}
        }}
    ) AS raw_response
""").collect()[0][0]

print("Raw Cortex response:")
print(raw_result)

### EXERCISE: Add Validation Rules

The basic schema above accepts any value that matches the type. But we know things about our domain -- states are 2-letter codes, water depth can't be negative, damage severity has a fixed set of values. Add Pydantic **field validators** to catch common extraction errors.

Your validators should check:
1. `state` must be exactly 2 uppercase letters
2. `water_depth_inches` must be >= 0 and <= 600 (50 feet max)
3. `building_damage_amount` and `contents_damage_amount` must be >= 0
4. `basement_type` must be between 0 and 4
5. `structural_damage_severity` must be one of: minor, moderate, severe, catastrophic (or null)
6. `number_of_stories` must be between 1 and 5 (or null)

In [ ]:
# EXERCISE: Add field validators to catch extraction errors.
# ClaimExtractionValidated inherits all 21 fields from ClaimExtraction --
# no need to re-declare them. Just add @field_validator methods.

class ClaimExtractionValidated(ClaimExtraction):
    """Adds domain-specific validation rules on top of the base extraction schema."""

    # --- PROVIDED: Two example validators showing the pattern ---

    @field_validator('state')
    @classmethod
    def validate_state(cls, v):
        # Pattern: check format, normalize to uppercase, reject clearly wrong values
        if v is not None and (len(v) != 2 or not v.isalpha()):
            raise ValueError(f"State must be a 2-letter code, got: {v!r}")
        return v.upper() if v else v

    @field_validator('structural_damage_severity')
    @classmethod
    def validate_structural_damage_severity(cls, v):
        # Pattern: enforce a fixed enum of allowed values and normalize case
        if v is not None:
            allowed = {'minor', 'moderate', 'severe', 'catastrophic'}
            if v.lower() not in allowed:
                raise ValueError(f"Severity must be one of {allowed}, got: {v!r}")
            return v.lower()
        return v

    # --- EXERCISE: Implement the three validators below ---

    @field_validator('water_depth_inches')
    @classmethod
    def validate_water_depth_inches(cls, v):
        # TODO: water depth can't be negative; values > 600 inches (50 ft) are likely errors
        return v

    @field_validator('building_damage_amount')
    @classmethod
    def validate_building_damage_amount(cls, v):
        # TODO: damage amounts must be non-negative
        return v

    @field_validator('basement_type')
    @classmethod
    def validate_basement_type(cls, v):
        # TODO: FEMA codes are 0–4 (0=none, 1=basement, 2=enclosure, 3=crawlspace, 4=subgrade)
        return v


# --- Test your validators ---
# This should pass:
valid_data = {"state": "TX", "water_depth_inches": 24, "structural_damage_severity": "moderate"}
print("Valid data test:", ClaimExtractionValidated(**valid_data))

# Uncomment to test your validators with invalid data:
# invalid_data = {"state": "TEXAS", "water_depth_inches": -5, "basement_type": 9}
# ClaimExtractionValidated(**invalid_data)

These Pydantic models serve double duty in our LangGraph pipeline:
- The **JSON schema** feeds into Cortex's `response_format` to constrain the LLM's output structure
- The **validators** run in the VALIDATE node to catch extraction errors before they reach your database

Notice we organized fields into three groups: **Claim & Location**, **Flood Event & Damage**, and **Property Details**. In the next section, we'll use these groups to build a pipeline where each group gets its own focused extraction prompt via LangGraph's **Send API**.

> **Day-to-day takeaway:** Your Pydantic model *is* your spec. Every field description you write becomes both the LLM's instruction and your team's documentation. Validators are the safety net — they catch the cases where the LLM hallucinated a value, misread a unit, or used a string where you needed a code. The fix loop in Section 3 only works because validators produce machine-readable error messages that the LLM can act on.

---
# Section 3: Building the Extraction Pipeline

Now we put it all together. Instead of cramming all 21 fields into a single prompt, we'll split extraction into **focused groups** using LangGraph's **Send API**. Each group gets its own Cortex call with a prompt tailored to just those fields -- and each group **validates and fixes its own output** before merging.

**Why group?** As schemas grow beyond ~15-20 fields, single-prompt extraction degrades -- the LLM juggles too many field definitions, confuses similar fields (e.g., building damage vs contents damage vs building payout), and you can't tune prompts per domain area. Grouping keeps each prompt focused and lets you add groups independently.

**Why validate per-group?** If validation happens after merging all groups, a single bad field forces re-extraction of *all* fields -- wasteful when two groups extracted perfectly. Per-group validation means only the failing group retries, saving cost and latency.

The pipeline architecture:

```
                    +─────────+
                    |  START   |
                    +────┬─────+
                         |
              +──────────┼──────────+
              |          |          |   Fan-out via Send()
        +─────v──+ +────v───+ +───v──────+
        | Group1 | | Group2 | | Group3   |
        |Claim & | |Flood & | |Property  |
        |Location| |Damage  | |Details   |
        +────┬───+ +────┬───+ +────┬─────+
             |          |          |     Per-group:
        +────v───+ +────v───+ +───v──────+
        |VALIDATE| |VALIDATE| |VALIDATE  |  extract → validate
        +────┬───+ +────┬───+ +────┬─────+     ↕ fix (retry loop)
             |          |          |
             +──────────┼──────────+
                   +────v─────+
                   |  MERGE   |   Combine validated group results
                   +────┬─────+
                   +────v─────+
                   | FINALIZE |   Package with metadata
                   +────┬─────+
                   +────v─────+
                   |   END    |
                   +──────────+
```

> **Note on parallelism**: In Snowpark notebooks, SQL calls are synchronous, so the three Send targets run sequentially. The Send pattern still matters because: (1) each group gets a focused prompt, (2) groups can have specialized instructions, and (3) in environments with async I/O the same graph runs in parallel without code changes.

In [ ]:
# Step 1: Define the graph state.
# We need three state types:
#   - ExtractionState: the main pipeline state
#   - WorkerOutput: controls which keys propagate from worker subgraph to parent
#   - GroupWorkerState: isolated state for each group's extract → validate → fix workflow

class ExtractionState(TypedDict):
    # --- Inputs (set once at invocation) ---
    claim_id: str                    # Unique identifier for tracking
    adjuster_notes: str              # Raw unstructured text to extract from
    max_retries: int                 # Maximum fix attempts per group
    
    # --- Group extraction results (fan-in from worker subgraphs via reducer) ---
    group_results: Annotated[list, operator.add]  # Each worker appends its validated group result
    
    # --- Merged results (set by merge node after all groups complete) ---
    validated_extraction: dict       # Combined validated extraction from all groups
    extraction_errors: list          # Any remaining validation errors across groups
    is_valid: bool                   # Whether ALL groups passed validation
    retry_count: int                 # Total fix attempts across all groups
    
    # --- Observability ---
    node_history: Annotated[list, operator.add]  # Workers append in parallel
    cortex_calls: int                # Total LLM calls made
    processing_time: float           # Total wall-clock seconds


class WorkerOutput(TypedDict):
    """Controls which keys propagate from worker subgraph to parent.
    Only group_results and node_history fan-in via reducers.
    All other worker state (extracted_fields, group_key, etc.) stays isolated."""
    group_results: Annotated[list, operator.add]
    node_history: Annotated[list, operator.add]


class GroupWorkerState(TypedDict):
    """State for each group's extract → validate → fix workflow.
    
    Each Send worker gets its own ISOLATED copy of this state via a subgraph.
    Only the keys in WorkerOutput are propagated back to the parent ExtractionState.
    """
    # --- Inputs (provided by Send) ---
    claim_id: str
    adjuster_notes: str
    group_key: str         # Which group this worker handles
    group_fields: list     # Field names in this group
    group_label: str       # Human-readable group name
    max_retries: int       # Maximum fix attempts for this group
    
    # --- Working state (updated by extract/validate/fix nodes) ---
    extracted_fields: dict       # Current extraction for this group
    extraction_errors: list      # Validation errors (if any)
    is_valid: bool               # Whether current extraction passes validation
    retry_count: int             # Number of fix attempts so far
    cortex_calls: int            # LLM calls made for this group
    processing_time: float       # Wall-clock seconds for this group
    
    # --- Output (merged into parent via reducers) ---
    group_results: Annotated[list, operator.add]
    node_history: Annotated[list, operator.add]

Notice the `Annotated[list, operator.add]` on `group_results` and `node_history`. This is LangGraph's **reducer** pattern -- when multiple Send workers return results simultaneously, the reducer tells LangGraph how to combine them. `operator.add` means "concatenate the lists," so each worker's results get merged automatically.

`GroupWorkerState` now carries everything a worker needs for its own extract → validate → fix loop. Each worker operates independently -- if the "Property Details" group needs a fix retry but "Claim & Location" validated on the first try, only the property group pays for the extra LLM call.

The observability fields (`node_history`, `cortex_calls`, `processing_time`) are critical for:
- **Cost tracking**: knowing how many LLM calls each claim required, and which groups triggered retries
- **Debugging**: seeing the exact path through the graph per group
- **Performance analysis**: measuring wall-clock time per claim and per group

### Step 2: Fan-out Dispatcher & Extract Node

The dispatcher creates one `Send()` per field group. Each worker's extract node builds a focused prompt containing only that group's field descriptions, then calls Cortex with the group-specific schema.

In [ ]:
# --- THE KEY INSIGHT: One Send() per group = parallel execution ---
# fan_out_to_groups fires one Send() per field group.
# LangGraph routes each Send() to an independent "group_worker" subgraph invocation.

def fan_out_to_groups(state: ExtractionState) -> list:
    """Route to one extraction worker per field group using LangGraph Send."""
    return [
        Send("group_worker", {
            "claim_id": state["claim_id"],
            "adjuster_notes": state["adjuster_notes"],
            "group_key": group_key,
            "group_fields": group_info["fields"],
            "group_label": group_info["label"],
            "max_retries": state.get("max_retries", 2),
            "group_results": [],
            "node_history": [],
        })
        for group_key, group_info in FIELD_GROUPS.items()
    ]

In [ ]:
# --- PROVIDED: Extraction node (builds prompt → calls Cortex → returns dict) ---
def extract_group_node(state: GroupWorkerState) -> dict:
    """Extract fields for a single group. Called once per group via Send."""
    start_time = time.time()
    group_key = state["group_key"]
    fields = state["group_fields"]
    label = state["group_label"]
    
    # Build a focused prompt for just this group's fields
    field_descriptions = "\n".join(
        f"- {f}: {ClaimExtractionValidated.model_fields[f].description}"
        for f in fields
    )
    
    extraction_prompt = json.dumps([
        {
            "role": "system",
            "content": (
                f"You are an expert insurance data extraction system. "
                f"Extract ONLY the following {label} fields from the adjuster notes. "
                f"Be precise with numbers, dates, and codes. "
                f"If a field is not mentioned or cannot be reasonably inferred from the notes, use null.\n\n"
                f"Fields to extract:\n{field_descriptions}"
            )
        },
        {"role": "user", "content": state["adjuster_notes"]}
    ])
    
    group_schema = json.dumps(schema_for_group(group_key))
    
    # Tag with group info for per-group cost tracking
    session.sql(
        f"ALTER SESSION SET QUERY_TAG = "
        f"'{{\"lab\": \"claims-extraction\", \"section\": \"extract\", "
        f"\"node\": \"extract_group\", \"group\": \"{group_key}\"}}'"
    ).collect()
    
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON($${extraction_prompt}$$),
            {{
                'temperature': 0,
                'response_format': {{
                    'type': 'json',
                    'schema': PARSE_JSON($${group_schema}$$)
                }}
            }}
        ):structured_output[0]:raw_message::STRING AS result
    """).collect()[0][0]
    
    extracted = json.loads(result)
    elapsed = time.time() - start_time
    
    non_null = sum(1 for v in extracted.values() if v is not None)
    print(f"    [{label}] {non_null}/{len(fields)} fields extracted ({elapsed:.1f}s)")
    
    return {
        "extracted_fields": extracted,
        "cortex_calls": 1,
        "processing_time": elapsed,
        "node_history": [f"extract_{group_key}"],
    }

### Step 3: Validate Node

Each worker validates only its group's fields using the Pydantic rules from Section 2 -- no LLM calls, just domain checks. Since all fields are `Optional`, passing a subset to the full model works without triggering validators for other groups.

In [ ]:
# --- PROVIDED: Validates each group's extracted fields using your Pydantic rules ---
def validate_group_node(state: GroupWorkerState) -> dict:
    """Validate a single group's extracted fields using Pydantic domain rules."""
    group_key = state["group_key"]
    fields = state["group_fields"]
    raw = state["extracted_fields"]
    errors = []
    
    try:
        validated = ClaimExtractionValidated(**raw)
        # Extract only this group's validated fields (validators may transform values)
        validated_fields = {f: getattr(validated, f) for f in fields}
        
        result = {
            "extracted_fields": validated_fields,
            "extraction_errors": [],
            "is_valid": True,
            "node_history": [f"validate_{group_key}_pass"],
        }
        
        # Package the validated group result for fan-in to merge
        result["group_results"] = [{
            "group_key": group_key,
            "fields": validated_fields,
            "is_valid": True,
            "retry_count": state.get("retry_count", 0),
            "cortex_calls": state.get("cortex_calls", 0),
            "processing_time": state.get("processing_time", 0),
            "errors": [],
        }]
        
        return result
        
    except Exception as e:
        if hasattr(e, 'errors'):
            for err in e.errors():
                field_name = ".".join(str(x) for x in err["loc"])
                # Only include errors for fields in THIS group
                if field_name in fields:
                    errors.append({
                        "field": field_name,
                        "message": err["msg"],
                        "type": err["type"],
                        "input_value": str(err.get("input", ""))[:100]
                    })
        else:
            errors.append({"field": "unknown", "message": str(e), "type": "general"})
        
        result = {
            "extraction_errors": errors,
            "is_valid": False,
            "node_history": [f"validate_{group_key}_fail"],
        }
        
        # If no group-specific errors found (errors were in other groups' fields),
        # treat this group as valid
        if not errors:
            result["is_valid"] = True
            result["node_history"] = [f"validate_{group_key}_pass"]
            result["group_results"] = [{
                "group_key": group_key,
                "fields": raw,
                "is_valid": True,
                "retry_count": state.get("retry_count", 0),
                "cortex_calls": state.get("cortex_calls", 0),
                "processing_time": state.get("processing_time", 0),
                "errors": [],
            }]
        # If retries exhausted, package result anyway so we don't block merge
        elif state.get("retry_count", 0) >= state.get("max_retries", 2):
            print(f"    [{state['group_label']}] Max retries reached, finalizing with errors")
            result["group_results"] = [{
                "group_key": group_key,
                "fields": raw,
                "is_valid": False,
                "retry_count": state.get("retry_count", 0),
                "cortex_calls": state.get("cortex_calls", 0),
                "processing_time": state.get("processing_time", 0),
                "errors": errors,
            }]
        
        return result

### Step 4: Fix Node

When validation fails, the fix node re-prompts Cortex with the specific errors and the original notes. It uses only the failing group's schema, making the retry cheaper and more focused than re-extracting all 21 fields.

In [ ]:
# --- PROVIDED: Re-prompts Cortex with specific error descriptions to fix a single group ---
def fix_group_node(state: GroupWorkerState) -> dict:
    """Re-prompt Cortex with specific validation errors to fix a single group."""
    start_time = time.time()
    group_key = state["group_key"]
    label = state["group_label"]
    fields = state["group_fields"]
    
    # Format the errors into a clear description for the LLM
    error_descriptions = "\n".join([
        f"- Field '{e['field']}': {e['message']} (current value: {e['input_value']})"
        for e in state["extraction_errors"]
    ])
    
    # Include field descriptions so the LLM knows the expected format
    field_descriptions = "\n".join(
        f"- {f}: {ClaimExtractionValidated.model_fields[f].description}"
        for f in fields
    )
    
    fix_prompt = json.dumps([
        {
            "role": "system", 
            "content": (
                f"You are an expert insurance data extraction system. "
                f"A previous extraction attempt for {label} fields had validation errors. "
                f"Fix ONLY the fields with errors. "
                f"Keep all correctly extracted fields unchanged.\n\n"
                f"Fields:\n{field_descriptions}"
            )
        },
        {
            "role": "user", 
            "content": (
                f"Original adjuster notes:\n{state['adjuster_notes']}\n\n"
                f"Previous extraction (with errors):\n{json.dumps(state['extracted_fields'], indent=2)}\n\n"
                f"Validation errors to fix:\n{error_descriptions}\n\n"
                "Please provide the corrected extraction for these fields only."
            )
        }
    ])
    
    # Use the GROUP schema, not the full schema -- fewer fields = cheaper + more focused
    group_schema = json.dumps(schema_for_group(group_key))
    
    session.sql(
        f"ALTER SESSION SET QUERY_TAG = "
        f"'{{\"lab\": \"claims-extraction\", \"section\": \"extract\", "
        f"\"node\": \"fix_group\", \"group\": \"{group_key}\"}}'"
    ).collect()
    
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON($${fix_prompt}$$),
            {{
                'temperature': 0,
                'response_format': {{
                    'type': 'json',
                    'schema': PARSE_JSON($${group_schema}$$)
                }}
            }}
        ):structured_output[0]:raw_message::STRING AS result
    """).collect()[0][0]
    
    extracted = json.loads(result)
    elapsed = time.time() - start_time
    
    print(f"    [{label}] Fix attempt {state.get('retry_count', 0) + 1} ({elapsed:.1f}s)")
    
    return {
        "extracted_fields": extracted,
        "retry_count": state.get("retry_count", 0) + 1,
        "cortex_calls": state.get("cortex_calls", 0) + 1,
        "processing_time": state.get("processing_time", 0) + elapsed,
        "node_history": [f"fix_{group_key}"],
    }

### Steps 5-6: Merge, Route & Finalize

Three supporting pieces:
- **`merge_groups_node`**: combines validated results from all group workers into one dict
- **`should_retry_group`**: routes each worker to either `fix_group` (retry) or `END` (done)
- **`finalize_node`**: attaches processing metadata to the merged result

In [ ]:
def merge_groups_node(state: ExtractionState) -> dict:
    """Merge validated results from all group workers into one extraction dict."""
    merged = {}
    total_calls = 0
    total_time = 0.0
    total_retries = 0
    all_valid = True
    all_errors = []
    
    for group_result in state.get("group_results", []):
        merged.update(group_result["fields"])
        total_calls += group_result.get("cortex_calls", 0)
        total_time += group_result.get("processing_time", 0)
        total_retries += group_result.get("retry_count", 0)
        if not group_result.get("is_valid", False):
            all_valid = False
        all_errors.extend(group_result.get("errors", []))
    
    return {
        "validated_extraction": merged,
        "is_valid": all_valid,
        "cortex_calls": total_calls,
        "processing_time": total_time,
        "retry_count": total_retries,
        "extraction_errors": all_errors,
        "node_history": ["merge"],
    }


def should_retry_group(state: GroupWorkerState) -> str:
    """Decide whether to retry this group's extraction or finish the worker."""
    if state.get("is_valid", False):
        return END
    elif state.get("retry_count", 0) < state.get("max_retries", 2):
        return "fix_group"
    else:
        # Max retries exhausted for this group -- finish with what we have
        return END


def finalize_node(state: ExtractionState) -> dict:
    """Finalize the extraction result with metadata."""
    result = state.get("validated_extraction", {})
    
    # Attach processing metadata to the result
    result["_metadata"] = {
        "claim_id": state["claim_id"],
        "is_valid": state.get("is_valid", False),
        "retry_count": state.get("retry_count", 0),
        "cortex_calls": state.get("cortex_calls", 0),
        "node_path": " -> ".join(state.get("node_history", []) + ["finalize"]),
        "processing_time_seconds": round(state.get("processing_time", 0), 2),
        "had_errors": len(state.get("extraction_errors", [])) > 0
    }
    
    return {
        "validated_extraction": result,
        "node_history": ["finalize"]
    }

### Step 7: Assemble the Graph

Wire everything together: a **worker subgraph** (extract → validate → fix loop) that runs per-group, and a **parent graph** that fans out to workers then merges and finalizes.

In [ ]:
# --- Worker subgraph ---
worker_builder = StateGraph(GroupWorkerState, output_schema=WorkerOutput)
worker_builder.add_node("extract_group", extract_group_node)
worker_builder.add_node("validate_group", validate_group_node)
worker_builder.add_node("fix_group", fix_group_node)

worker_builder.add_edge(START, "extract_group")
worker_builder.add_edge("extract_group", "validate_group")
worker_builder.add_conditional_edges(
    "validate_group",
    should_retry_group,
    {"fix_group": "fix_group", END: END}
)
worker_builder.add_edge("fix_group", "validate_group")
worker_subgraph = worker_builder.compile()

# --- Parent graph ---
workflow = StateGraph(ExtractionState)
workflow.add_node("group_worker", worker_subgraph)
workflow.add_node("merge", merge_groups_node)
workflow.add_node("finalize", finalize_node)

workflow.add_conditional_edges(
    START,
    fan_out_to_groups,
    ["group_worker"]
)

workflow.add_edge("group_worker", "merge")
workflow.add_edge("merge", "finalize")
workflow.add_edge("finalize", END)

extraction_graph = workflow.compile()

print("Per-group extraction pipeline compiled!")
print(f"Groups: {[g['label'] for g in FIELD_GROUPS.values()]}")
print(f"Flow: START -> fan_out(3 groups) -> [extract -> validate -> fix loop] per group -> merge -> finalize")


In [ ]:

from IPython.display import Image, display

display(Image(extraction_graph.get_graph().draw_mermaid_png()))


### Test: Run the Full Pipeline

Let's test the complete graph on a single claim. Watch for the per-group extraction logs -- you should see each group extract independently, then merge into the final result.

In [ ]:
test_claim = session.sql(
    "SELECT claim_id, adjuster_notes FROM CLAIMS_SAMPLE LIMIT 1"
).collect()[0]

initial_state = {
    "claim_id": str(test_claim[0]),
    "adjuster_notes": test_claim[1],
    "max_retries": 2,
    # Defaults for reducer and tracking fields
    "group_results": [], "validated_extraction": {}, "extraction_errors": [],
    "retry_count": 0, "is_valid": False, "node_history": [],
    "cortex_calls": 0, "processing_time": 0.0,
}

print(f"Processing claim {initial_state['claim_id']}...")
print(f"Note preview: {initial_state['adjuster_notes'][:200]}...\n")
print(f"Extracting {len(ALL_FIELDS)} fields across {len(FIELD_GROUPS)} groups...\n")

result = extraction_graph.invoke(initial_state)

print(f"\nNode path: {' -> '.join(result['node_history'])}")
print(f"Valid: {result['is_valid']}")
print(f"Cortex calls: {result['cortex_calls']} ({len(FIELD_GROUPS)} extract + {result['retry_count']} fix retries)")
print(f"Processing time: {result['processing_time']:.2f}s")

# Per-group breakdown
print(f"\nPer-group results:")
for gr in result['group_results']:
    status = "VALID" if gr['is_valid'] else "INVALID"
    print(f"    [{gr['group_key']}] {status} ({gr['cortex_calls']} calls, {gr['retry_count']} retries)")

# Show extracted data with null/non-null summary
extracted = result['validated_extraction']
non_null = sum(1 for k, v in extracted.items() if v is not None and k != '_metadata')
print(f"\nExtracted {non_null}/{len(ALL_FIELDS)} fields (non-null):")
print(json.dumps({k: v for k, v in extracted.items() if k != '_metadata'}, indent=2))

### Section 3 Recap

We built a grouped extraction pipeline with per-group validation using LangGraph's Send API:

- **Fan-out**: `fan_out_to_groups` dispatches one `Send()` per field group, each with a focused prompt
- **Per-group validate + fix**: each worker validates its own output and retries only if *its* fields have errors -- no wasted re-extraction of passing groups
- **Merge**: `merge_groups_node` combines already-validated results from all groups into a single extraction dict
- **Observability**: every node logs its activity via `node_history`, and per-group Cortex calls are tracked via query tags

The grouped approach trades more Cortex calls for more focused prompts. Each group's prompt only asks about 5-10 fields instead of all 21, reducing confusion between similar fields (e.g., `building_damage_amount` vs `amount_paid_building`). And per-group validation means fix retries are cheaper -- only the failing group's schema is re-prompted, not the full 21-field schema.

> **Day-to-day takeaway:** The `fan_out → [extract/validate/fix] × N → merge` pattern is the reusable skeleton for any grouped extraction job. Swap in a different Pydantic schema and a different `FIELD_GROUPS` dict and the entire pipeline runs unchanged. Adding a new field group is one dict entry — no graph rewiring. And per-group retry means a bad `flood_damage` extraction doesn't force re-extraction of `claim_location` fields that already validated cleanly.

---
# Section 4: Running Extraction at Scale

Processing a single claim is straightforward. Processing hundreds or thousands requires careful engineering:

- **Progress tracking** -- know what's been processed so you can resume after interruptions
- **Error isolation** -- one bad claim shouldn't kill the entire batch
- **Result persistence** -- save results to Snowflake tables, not just Python variables
- **Idempotency** -- re-running a cell shouldn't create duplicates

> **Snowflake Feature**: We'll use `MERGE` statements for idempotent progress tracking and `VARIANT` columns to store the extracted JSON alongside metadata.

### Persistence Tables

We need two tables: `EXTRACTION_RESULTS` stores the extracted JSON and metadata, and `EXTRACTION_PROGRESS` tracks which claims have been processed for restartability.

In [ ]:
-- Create tables for results and progress tracking.
-- Using IF NOT EXISTS makes this safe to re-run after a kernel restart.

CREATE TABLE IF NOT EXISTS EXTRACTION_RESULTS (
    claim_id STRING,
    extracted_data VARIANT,
    is_valid BOOLEAN,
    retry_count INTEGER,
    cortex_calls INTEGER,
    processing_time_seconds FLOAT,
    node_path STRING,
    extraction_errors VARIANT,
    extracted_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS EXTRACTION_PROGRESS (
    claim_id STRING PRIMARY KEY,
    status STRING,          -- 'completed', 'failed', 'in_progress'
    started_at TIMESTAMP_NTZ,
    completed_at TIMESTAMP_NTZ
);

SELECT 'Tables ready' AS status,
    (SELECT COUNT(*) FROM EXTRACTION_RESULTS) AS existing_results,
    (SELECT COUNT(*) FROM EXTRACTION_PROGRESS) AS existing_progress;

### Batch Extraction with Checkpointing

The batch function queries for unprocessed claims, processes each through the graph, and saves results after every claim. If the kernel restarts, re-running picks up where it left off thanks to the progress table.

In [ ]:
def run_batch_extraction(batch_size=10, max_retries=2):
    """Run extraction on unprocessed claims with per-claim checkpointing."""
    
    # Find claims that haven't been processed yet -- this is the restartability key
    unprocessed = session.sql(f"""
        SELECT fc.claim_id, fc.adjuster_notes
        FROM FIMA_CLAIMS fc
        LEFT JOIN EXTRACTION_PROGRESS ep ON fc.claim_id::STRING = ep.claim_id
        WHERE ep.claim_id IS NULL OR ep.status = 'failed'
        LIMIT {batch_size}
    """).collect()
    
    total = len(unprocessed)
    if total == 0:
        print("All claims already processed! Nothing to do.")
        return [], []
    
    print(f"Processing {total} unprocessed claims (batch_size={batch_size})...\n")
    results = []
    errors = []
    
    for i, row in enumerate(unprocessed):
        claim_id = str(row[0])
        notes = row[1]
        print(f"  [{i+1}/{total}] Claim {claim_id}...", flush=True)
        
        # Mark as in-progress using MERGE (idempotent)
        session.sql(f"""
            MERGE INTO EXTRACTION_PROGRESS ep
            USING (SELECT '{claim_id}' AS claim_id) src
            ON ep.claim_id = src.claim_id
            WHEN MATCHED THEN UPDATE SET status='in_progress', started_at=CURRENT_TIMESTAMP()
            WHEN NOT MATCHED THEN INSERT (claim_id, status, started_at)
                VALUES (src.claim_id, 'in_progress', CURRENT_TIMESTAMP())
        """).collect()
        
        try:
            state = {
                "claim_id": claim_id,
                "adjuster_notes": notes,
                "group_results": [],
                "validated_extraction": {},
                "extraction_errors": [],
                "max_retries": max_retries,
                "retry_count": 0,
                "is_valid": False,
                "node_history": [],
                "cortex_calls": 0,
                "processing_time": 0.0
            }
            
            result = extraction_graph.invoke(state)
            
            # Save result to table
            extracted_json = json.dumps(result['validated_extraction']).replace("'", "''")
            errors_json = json.dumps(result.get('extraction_errors', [])).replace("'", "''")
            node_path = ' -> '.join(result['node_history'])
            
            session.sql(f"""
                INSERT INTO EXTRACTION_RESULTS
                (claim_id, extracted_data, is_valid, retry_count, cortex_calls, 
                 processing_time_seconds, node_path, extraction_errors)
                SELECT
                    '{claim_id}',
                    PARSE_JSON('{extracted_json}'),
                    {result['is_valid']},
                    {result['retry_count']},
                    {result['cortex_calls']},
                    {result['processing_time']:.2f},
                    '{node_path}',
                    PARSE_JSON('{errors_json}')
            """).collect()
            
            # Mark completed
            session.sql(f"""
                UPDATE EXTRACTION_PROGRESS
                SET status='completed', completed_at=CURRENT_TIMESTAMP()
                WHERE claim_id='{claim_id}'
            """).collect()
            
            status = "VALID" if result['is_valid'] else "INVALID"
            print(f"    {status} ({result['cortex_calls']} calls, {result['retry_count']} retries, {result['processing_time']:.1f}s)")
            results.append(result)
            
        except Exception as e:
            print(f"    FAILED: {str(e)[:80]}")
            session.sql(f"""
                UPDATE EXTRACTION_PROGRESS
                SET status='failed', completed_at=CURRENT_TIMESTAMP()
                WHERE claim_id='{claim_id}'
            """).collect()
            errors.append({"claim_id": claim_id, "error": str(e)})
    
    print(f"\nBatch complete: {len(results)} succeeded, {len(errors)} failed")
    return results, errors

In [ ]:
# Run a small initial batch to verify everything works end-to-end
results, errors = run_batch_extraction(batch_size=3, max_retries=2)

### EXPERIMENT: Batch Size

Try running with `batch_size=20` in the cell below. Does it take roughly twice as long as the batch of 10? Why or why not?

Consider: each claim is processed **sequentially** in our Python loop. What would happen if we could parallelize? (We'll explore this in the Performance section.)

In [ ]:
# Process the remaining claims. Adjust batch_size based on your time budget.
# Because of restartability, this picks up where the previous batch left off.
results, errors = run_batch_extraction(batch_size=20, max_retries=2)

### Review Extraction Results

Now let's check how the extraction went -- overall progress, and a preview of the extracted fields using Snowflake's `VARIANT` column notation.

In [ ]:
-- Check overall extraction progress
SELECT
    status,
    COUNT(*) AS claim_count,
    ROUND(AVG(DATEDIFF('second', started_at, completed_at)), 1) AS avg_duration_seconds
FROM EXTRACTION_PROGRESS
GROUP BY status
ORDER BY claim_count DESC;

In [ ]:
-- Preview extracted fields using VARIANT : notation
SELECT
    claim_id,
    is_valid,
    retry_count,
    cortex_calls,
    ROUND(processing_time_seconds, 1) AS time_sec,
    node_path,
    extracted_data:state::STRING AS extracted_state,
    extracted_data:building_damage_amount::FLOAT AS extracted_bldg_damage,
    extracted_data:water_depth_inches::INTEGER AS extracted_water_depth,
    extracted_data:flood_zone::STRING AS extracted_flood_zone
FROM EXTRACTION_RESULTS
ORDER BY extracted_at DESC
LIMIT 10;

> **Day-to-day takeaway:** The idempotent `MERGE` pattern (insert if new, update if exists, skip if already processed) is what makes this safe to run repeatedly. Restart after a crash, add new claims to the source table, or reprocess a bad batch — the query always does the right thing without duplicating rows. This pattern scales from 50 records in a lab to 50,000 records in production without changing a line of code.

---
# Section 5: Cost, Performance & Scaling Notes

The sections above covered extraction pipeline design and accuracy. In production, you'd also need to think about **cost**, **performance**, and **scaling**. Here's a quick reference.

### Cost Tracking with Query Tags

Throughout this lab, every Cortex call was tagged with structured JSON metadata via `ALTER SESSION SET QUERY_TAG`. This lets you query `INFORMATION_SCHEMA.QUERY_HISTORY` to attribute costs by section, node, group, etc. -- all without external tooling.

Key takeaways:
- **`temperature=0`** reduces retries, saving credits
- **Structured output** (`response_format`) eliminates JSON parsing failures entirely
- **The retry pattern trades cost for quality** -- track retry rates to find the balance
- **Grouped extraction costs more per claim** (~3 extract calls vs 1) but improves accuracy on complex schemas by keeping prompts focused

### Performance: Python Loop vs SQL Batch vs Hybrid

| Approach | How It Works | Validation | Best For |
|----------|-------------|------------|----------|
| **Python loop** (this lab) | Sequential, one claim at a time | Full Pydantic + retry | Small batches, max quality |
| **SQL batch** (`SELECT ... CORTEX.COMPLETE ... FROM table`) | Snowflake parallelizes across rows | None (raw JSON output) | Max throughput, simple schemas |
| **Hybrid** | SQL batch extract, then Python validate, then LangGraph fix only failures | Full Pydantic, retry only failures | **Production** -- fast AND reliable |

### Scaling to 2.7M Claims

- **SQL CTAS**: `CREATE TABLE AS SELECT` with `CORTEX.COMPLETE` -- Snowflake handles parallelism automatically
- **Warehouse sizing**: Larger warehouses = more parallel Cortex calls = higher throughput (but same total credits)
- **Multi-cluster warehouses** (Enterprise Edition): Auto-scale out with multiple clusters for concurrent queries
- **Partitioned processing**: Split by state or year, process partitions independently for natural checkpointing and cost attribution

In [ ]:
-- Example: Cost breakdown by pipeline section and group.
-- Run this after processing a batch to see where credits went.
SELECT
    PARSE_JSON(query_tag):section::STRING AS lab_section,
    PARSE_JSON(query_tag):node::STRING AS graph_node,
    PARSE_JSON(query_tag):group::STRING AS field_group,
    COUNT(*) AS query_count,
    ROUND(SUM(credits_used_cloud_services), 4) AS total_credits,
    ROUND(AVG(total_elapsed_time) / 1000, 1) AS avg_time_seconds
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    DATE_RANGE_START => DATEADD('hour', -6, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10000
))
WHERE query_tag LIKE '%claims-extraction%'
  AND query_text ILIKE '%CORTEX%COMPLETE%'
GROUP BY lab_section, graph_node, field_group
ORDER BY total_credits DESC;

### What's Next

In the **Model Evaluation lab**, you'll build a comprehensive evaluation framework on top of the `EXTRACTION_RESULTS` and `CLAIMS_SAMPLE` tables created here. That lab covers precision/recall, confusion matrices, regression metrics, semantic similarity, and systematic error analysis.


---
### Bonus: Stored Procedure (Optional)

> **Skip if short on time.** The cell below packages everything you've already built into a Snowflake stored procedure so it can run on a schedule or be called from other pipelines. The logic is identical to the batch extraction loop above — it's the same extract → validate → fix → MERGE pattern, just wrapped in SQL `CALL EXTRACT_CLAIMS_PARALLEL(...)`.
>
> Read it if you're curious how Python-based LangGraph pipelines deploy as serverless Snowflake tasks. Otherwise, move on to the **Model Evaluation lab**.

**What the stored procedure adds:**
- Accepts `BATCH_SIZE`, `MAX_WORKERS`, and `MAX_RETRIES` as parameters
- Can be scheduled with `CREATE TASK` for nightly runs
- Logs execution metadata to the same `EXTRACTION_PROGRESS` table you already built

In [ ]:
session.sql("""
CREATE OR REPLACE PROCEDURE EXTRACT_CLAIMS_PARALLEL(
    BATCH_SIZE INT,
    MAX_WORKERS INT,
    MAX_RETRIES INT
)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'pydantic')
HANDLER = 'run'
EXECUTE AS CALLER
AS
$$
import json
import time
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel, Field, field_validator


# ── Schema & field groups (same as notebook) ──────────────────────────────

class ClaimExtractionValidated(BaseModel):
    claim_id: Optional[str] = Field(None, description="The claim identifier or number")
    date_of_loss: Optional[str] = Field(None, description="Date when the flood damage occurred (YYYY-MM-DD format)")
    state: Optional[str] = Field(None, description="Two-letter US state code")
    county_code: Optional[str] = Field(None, description="County FIPS code")
    flood_zone: Optional[str] = Field(None, description="FEMA flood zone designation (e.g., AE, V, X, A)")
    flood_event: Optional[str] = Field(None, description="Name of the flood event or disaster")
    water_depth_inches: Optional[int] = Field(None, description="Depth of flood water in inches")
    flood_duration_hours: Optional[int] = Field(None, description="How long flood water remained, in hours")
    building_damage_amount: Optional[float] = Field(None, description="Dollar amount of building/structural damage")
    contents_damage_amount: Optional[float] = Field(None, description="Dollar amount of contents/personal property damage")
    building_property_value: Optional[float] = Field(None, description="Assessed or estimated value of the building")
    contents_property_value: Optional[float] = Field(None, description="Assessed or estimated value of contents")
    structural_damage_severity: Optional[str] = Field(None, description="Overall severity: minor, moderate, severe, or catastrophic")
    amount_paid_building: Optional[float] = Field(None, description="Insurance payout amount for building damage")
    amount_paid_contents: Optional[float] = Field(None, description="Insurance payout amount for contents damage")
    basement_type: Optional[int] = Field(None, description="Basement/enclosure/crawlspace type code (0-4)")
    floodproofed: Optional[bool] = Field(None, description="Whether the building has floodproofing measures")
    mold_present: Optional[bool] = Field(None, description="Whether mold was observed during inspection")
    number_of_stories: Optional[int] = Field(None, description="Number of stories in the building")
    elevation_certificate: Optional[bool] = Field(None, description="Whether an elevation certificate exists for the property")
    building_description_code: Optional[str] = Field(None, description="Building description type code")

    @field_validator('state')
    @classmethod
    def validate_state(cls, v):
        if v is not None and (len(v) != 2 or not v.isalpha()):
            raise ValueError(f"State must be 2 letters, got: {v}")
        return v.upper() if v else v

    @field_validator('water_depth_inches')
    @classmethod
    def validate_water_depth(cls, v):
        if v is not None and v < 0:
            raise ValueError(f"Water depth can't be negative, got: {v}")
        return v

    @field_validator('building_damage_amount', 'contents_damage_amount')
    @classmethod
    def validate_damage_amount(cls, v):
        if v is not None and v < 0:
            raise ValueError(f"Damage amount can't be negative, got: {v}")
        return v

    @field_validator('structural_damage_severity')
    @classmethod
    def validate_severity(cls, v):
        if v is not None:
            allowed = {'minor', 'moderate', 'severe', 'catastrophic'}
            if v.lower() not in allowed:
                raise ValueError(f"Severity must be one of {allowed}, got: {v}")
            return v.lower()
        return v

    @field_validator('number_of_stories')
    @classmethod
    def validate_stories(cls, v):
        if v is not None and (v < 1 or v > 5):
            raise ValueError(f"Stories must be 1-5, got: {v}")
        return v


FIELD_GROUPS = {
    "claim_location": {
        "label": "Claim & Location",
        "fields": ["claim_id", "date_of_loss", "state", "county_code", "flood_zone"],
    },
    "flood_damage": {
        "label": "Flood Event & Damage",
        "fields": [
            "flood_event", "water_depth_inches", "flood_duration_hours",
            "building_damage_amount", "contents_damage_amount",
            "building_property_value", "contents_property_value",
            "structural_damage_severity", "amount_paid_building", "amount_paid_contents",
        ],
    },
    "property_details": {
        "label": "Property Details",
        "fields": [
            "basement_type", "floodproofed", "mold_present",
            "number_of_stories", "elevation_certificate", "building_description_code",
        ],
    },
}

ALL_FIELDS = [f for g in FIELD_GROUPS.values() for f in g["fields"]]


def simplify_schema_for_cortex(schema):
    simplified = {"type": "object", "properties": {}}
    for field_name, field_def in schema.get("properties", {}).items():
        prop = {}
        if "anyOf" in field_def:
            types = [opt["type"] for opt in field_def["anyOf"] if "type" in opt]
            prop["type"] = types if len(types) > 1 else types[0]
        elif "type" in field_def:
            prop["type"] = field_def["type"]
        if "description" in field_def:
            prop["description"] = field_def["description"]
        simplified["properties"][field_name] = prop
    return simplified


def schema_for_group(group_key):
    full_schema = ClaimExtractionValidated.model_json_schema()
    group_fields = FIELD_GROUPS[group_key]["fields"]
    full_simplified = simplify_schema_for_cortex(full_schema)
    return {
        "type": "object",
        "properties": {
            k: v for k, v in full_simplified["properties"].items()
            if k in group_fields
        }
    }


# ── Per-group extraction with validate/fix loop ──────────────────────────

def extract_group(session, adjuster_notes, group_key, max_retries):
    group_info = FIELD_GROUPS[group_key]
    fields = group_info["fields"]
    label = group_info["label"]
    cortex_calls = 0
    start_time = time.time()

    # Build extraction prompt
    field_descriptions = "\\n".join(
        f"- {f}: {ClaimExtractionValidated.model_fields[f].description}"
        for f in fields
    )
    extraction_prompt = json.dumps([
        {
            "role": "system",
            "content": (
                f"You are an expert insurance data extraction system. "
                f"Extract ONLY the following {label} fields from the adjuster notes. "
                f"Be precise with numbers, dates, and codes. "
                f"If a field is not mentioned or cannot be reasonably inferred, use null.\\n\\n"
                f"Fields to extract:\\n{field_descriptions}"
            )
        },
        {"role": "user", "content": adjuster_notes}
    ])
    group_schema = json.dumps(schema_for_group(group_key))

    # Initial extraction
    result = session.sql(
        '''SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON(?),
            {
                'temperature': 0,
                'response_format': {
                    'type': 'json',
                    'schema': PARSE_JSON(?)
                }
            }
        ):structured_output[0]:raw_message::STRING AS result''',
        [extraction_prompt, group_schema]).collect()[0][0]
    extracted = json.loads(result)
    cortex_calls += 1

    # Validate → fix loop
    retry_count = 0
    errors = []
    is_valid = False

    for attempt in range(max_retries + 1):
        try:
            validated = ClaimExtractionValidated(**extracted)
            extracted = {f: getattr(validated, f) for f in fields}
            is_valid = True
            errors = []
            break
        except Exception as e:
            errors = []
            if hasattr(e, 'errors'):
                for err in e.errors():
                    field_name = ".".join(str(x) for x in err["loc"])
                    if field_name in fields:
                        errors.append({
                            "field": field_name,
                            "message": err["msg"],
                            "input_value": str(err.get("input", ""))[:100]
                        })
            if not errors:
                # Errors were in other groups' fields -- this group is fine
                is_valid = True
                break
            if attempt < max_retries:
                # Build fix prompt
                error_desc = "\\n".join(
                    f"- Field '{e['field']}': {e['message']} (current: {e['input_value']})"
                    for e in errors
                )
                fix_prompt = json.dumps([
                    {
                        "role": "system",
                        "content": (
                            f"You are an expert insurance data extraction system. "
                            f"A previous extraction for {label} fields had validation errors. "
                            f"Fix ONLY the fields with errors. Keep correct fields unchanged.\\n\\n"
                            f"Fields:\\n{field_descriptions}"
                        )
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Original adjuster notes:\\n{adjuster_notes}\\n\\n"
                            f"Previous extraction:\\n{json.dumps(extracted, indent=2)}\\n\\n"
                            f"Validation errors:\\n{error_desc}\\n\\n"
                            "Please provide the corrected extraction."
                        )
                    }
                ])
                result = session.sql(
                    '''SELECT SNOWFLAKE.CORTEX.COMPLETE(
                        'claude-3-5-sonnet',
                        PARSE_JSON(?),
                        {
                            'temperature': 0,
                            'response_format': {
                                'type': 'json',
                                'schema': PARSE_JSON(?)
                            }
                        }
                    ):structured_output[0]:raw_message::STRING AS result''',
                    [fix_prompt, group_schema]).collect()[0][0]
                extracted = json.loads(result)
                cortex_calls += 1
                retry_count += 1

    elapsed = time.time() - start_time
    return {
        "group_key": group_key,
        "fields": extracted,
        "is_valid": is_valid,
        "retry_count": retry_count,
        "cortex_calls": cortex_calls,
        "processing_time": round(elapsed, 2),
        "errors": errors,
    }


# ── Process one claim with parallel group extraction ──────────────────────

def process_claim(session, claim_id, adjuster_notes, max_retries):
    start_time = time.time()

    # Fan out field groups in parallel (inner parallelism)
    group_results = []
    with ThreadPoolExecutor(max_workers=len(FIELD_GROUPS)) as group_pool:
        futures = {
            group_pool.submit(extract_group, session, adjuster_notes, gk, max_retries): gk
            for gk in FIELD_GROUPS
        }
        for future in as_completed(futures):
            group_results.append(future.result())

    # Merge results
    merged = {}
    total_calls = 0
    total_retries = 0
    all_valid = True
    all_errors = []
    for gr in group_results:
        merged.update(gr["fields"])
        total_calls += gr["cortex_calls"]
        total_retries += gr["retry_count"]
        if not gr["is_valid"]:
            all_valid = False
        all_errors.extend(gr["errors"])

    elapsed = round(time.time() - start_time, 2)

    # Add metadata
    merged["_metadata"] = {
        "claim_id": claim_id,
        "is_valid": all_valid,
        "retry_count": total_retries,
        "cortex_calls": total_calls,
        "processing_time_seconds": elapsed,
    }

    # Save results
    extracted_json = json.dumps(merged).replace("'", "''")
    errors_json = json.dumps(all_errors).replace("'", "''")
    node_path = " | ".join(f"{gr['group_key']}({gr['cortex_calls']}calls)" for gr in group_results)

    session.sql(f\"\"\"
        INSERT INTO EXTRACTION_RESULTS
        (claim_id, extracted_data, is_valid, retry_count, cortex_calls,
         processing_time_seconds, node_path, extraction_errors)
        SELECT
            '{claim_id}',
            PARSE_JSON('{extracted_json}'),
            {all_valid},
            {total_retries},
            {total_calls},
            {elapsed},
            '{node_path}',
            PARSE_JSON('{errors_json}')
    \"\"\").collect()

    session.sql(f\"\"\"
        UPDATE EXTRACTION_PROGRESS
        SET status='completed', completed_at=CURRENT_TIMESTAMP()
        WHERE claim_id='{claim_id}'
    \"\"\").collect()

    return {
        "claim_id": claim_id,
        "is_valid": all_valid,
        "cortex_calls": total_calls,
        "retry_count": total_retries,
        "processing_time": elapsed,
    }


# ── Main entry point ─────────────────────────────────────────────────────

def run(session, batch_size, max_workers, max_retries):
    start_time = time.time()

    # Fetch unprocessed claims
    unprocessed = session.sql(f\"\"\"
        SELECT an.claim_id, an.adjuster_notes
        FROM FIMA_CLAIMS an
        LEFT JOIN EXTRACTION_PROGRESS ep ON an.claim_id::STRING = ep.claim_id
        WHERE ep.claim_id IS NULL OR ep.status = 'failed'
        LIMIT {batch_size}
    \"\"\").collect()

    total = len(unprocessed)
    if total == 0:
        return {"status": "complete", "message": "All claims already processed", "processed": 0}

    # Mark all as in-progress
    for row in unprocessed:
        claim_id = str(row[0])
        session.sql(f\"\"\"
            MERGE INTO EXTRACTION_PROGRESS ep
            USING (SELECT '{claim_id}' AS claim_id) src
            ON ep.claim_id = src.claim_id
            WHEN MATCHED THEN UPDATE SET status='in_progress', started_at=CURRENT_TIMESTAMP()
            WHEN NOT MATCHED THEN INSERT (claim_id, status, started_at)
                VALUES (src.claim_id, 'in_progress', CURRENT_TIMESTAMP())
        \"\"\").collect()

    # Process claims in parallel (outer parallelism)
    results = []
    errors = []
    with ThreadPoolExecutor(max_workers=max_workers) as claim_pool:
        futures = {}
        for row in unprocessed:
            claim_id = str(row[0])
            notes = row[1]
            future = claim_pool.submit(process_claim, session, claim_id, notes, max_retries)
            futures[future] = claim_id

        for future in as_completed(futures):
            claim_id = futures[future]
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                session.sql(f\"\"\"
                    UPDATE EXTRACTION_PROGRESS
                    SET status='failed', completed_at=CURRENT_TIMESTAMP()
                    WHERE claim_id='{claim_id}'
                \"\"\").collect()
                errors.append({"claim_id": claim_id, "error": str(e)[:200]})

    total_time = round(time.time() - start_time, 2)
    total_calls = sum(r["cortex_calls"] for r in results)
    avg_time = round(sum(r["processing_time"] for r in results) / len(results), 2) if results else 0

    return {
        "status": "complete",
        "processed": len(results),
        "failed": len(errors),
        "total_cortex_calls": total_calls,
        "wall_clock_seconds": total_time,
        "avg_claim_time_seconds": avg_time,
        "errors": errors if errors else None,
    }
$$;
""").collect()

print("Stored procedure EXTRACT_CLAIMS_PARALLEL created successfully!")

In [ ]:
-- Reset progress so we can test from scratch
-- (Only clears progress tracking, not results -- so you can compare runs)
TRUNCATE TABLE EXTRACTION_PROGRESS;

SELECT 'Progress table cleared' AS status,
    (SELECT COUNT(*) FROM EXTRACTION_RESULTS) AS existing_results;

In [ ]:
-- Test 1: Sequential baseline (max_workers=1)
-- Processes 5 claims one at a time, with parallel field-group extraction within each claim
CALL EXTRACT_CLAIMS_PARALLEL(5, 5, 2);

In [ ]:
select * from EXTRACTION_PROGRESS order by completed_at;